In [1]:
from MLstatkit.stats import Delong_test

import pandas as pd
import numpy as np
output_pkl = '/home/server/Projects/data/AKI/results/tabular_preop.pkl'
df_preop = pd.read_pickle(output_pkl)[['model_name', 'y_prob']]
output_pkl = '/home/server/Projects/data/AKI/results/tabular_combined.pkl'
df_combined = pd.read_pickle(output_pkl)[['model_name', 'y_prob']]

/home/server/.pyenv/versions/.venv3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df_combined['model_name'] = 'combined_' + df_combined['model_name']
df_preop['model_name'] = 'preop_' + df_preop['model_name']
dfs = [df_combined, df_preop]
df = pd.concat(dfs, axis=0, ignore_index=True)
model_names = df['model_name'].values.tolist()


In [ ]:
result = pd.DataFrame(index=model_names, columns=model_names)
key = df[df['model_name'] == 'combined_base']['y_pred_binary'].values[0]
for i in model_names:
    first = df.loc[df['model_name'] == i, 'y_prob'].values[0]
    for j in model_names:
        if i > j:
            second = df.loc[df['model_name'] == j, 'y_prob'].values[0]
            avg_p = np.mean([Delong_test(key[k], first[k], second[k])[1] for k in range(25)])
            result.loc[i, j] = avg_p
            result.loc[j, i] = avg_p

/home/server/.pyenv/versions/.venv3.9/lib/python3.9/site-packages/MLstatkit/stats.py:85: RuntimeWarning: invalid value encountered in divide
  z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, delongcov), l.T)).flatten()


In [11]:
result.to_csv('delong.csv')

In [1]:
# format delong values for supplemental table
import pandas as pd

df = pd.read_csv('delong.csv')

In [2]:
df = df.set_index('Unnamed: 0')
base_strings = [col for col in df.columns if 'base' in col]
df = df.drop(base_strings, axis=0)
df = df.drop(base_strings, axis=1)

In [3]:
def asterisks(a):
    if not a:
        return "N/A"
    if a < 0.0001:
        return f"p < 0.0001***"
    elif a < 0.001:
        return f"{a:.3g}**"
    elif a < 0.01:
        return f"{a:.3g}*"
    else:
        return f"{a:.3g}"

In [4]:
df_translated = df.map(asterisks)
print(df_translated.to_csv(sep='\t', index=True)) 

Unnamed: 0	combined_log_reg	combined_lin_reg	combined_autogluon	combined_xgb	combined_svm	combined_mlp	combined_rf	combined_knn	preop_log_reg	preop_lin_reg	preop_autogluon	preop_xgb	preop_svm	preop_mlp	preop_rf	preop_knn
combined_log_reg	nan	p < 0.0001***	0.423	0.108	p < 0.0001***	p < 0.0001***	0.409	p < 0.0001***	0.0448	p < 0.0001***	0.00493*	0.0482	p < 0.0001***	0.0055*	0.479	p < 0.0001***
combined_lin_reg	p < 0.0001***	nan	0.000628**	p < 0.0001***	0.000113**	p < 0.0001***	0.000174**	0.000288**	0.00247*	0.394	0.211	0.0921	p < 0.0001***	0.397	p < 0.0001***	p < 0.0001***
combined_autogluon	0.423	0.000628**	nan	0.18	p < 0.0001***	p < 0.0001***	0.401	p < 0.0001***	0.0835	p < 0.0001***	0.00268*	0.0746	p < 0.0001***	0.00254*	0.448	p < 0.0001***
combined_xgb	0.108	p < 0.0001***	0.18	nan	p < 0.0001***	p < 0.0001***	0.181	p < 0.0001***	0.00253*	p < 0.0001***	p < 0.0001***	0.0016*	p < 0.0001***	p < 0.0001***	0.152	p < 0.0001***
combined_svm	p < 0.0001***	0.000113**	p < 0.0001***	p < 0.0001***	